# Chapter 6.6: LLM as Ranker

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand** how LLMs can be used as zero-shot and few-shot rankers
2. **Implement** pointwise, pairwise, and listwise LLM ranking approaches
3. **Build** a Chat-Rec style conversational ranking system
4. **Analyze** calibration challenges in LLM-based rankers
5. **Diagnose** position bias (primacy/recency effects) in LLM ranking
6. **Compare** different LLM ranking strategies on synthetic benchmarks

## Prerequisites

- Understanding of learning-to-rank (pointwise, pairwise, listwise)
- Familiarity with prompt engineering
- Chapters 6.4-6.5 (LLM foundations and features for rec)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part6/chapter_6.6_llm_ranker.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/rec_system/main/notebooks/part6/chapter_6.6_llm_ranker.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cpu')
print(f'Using device: {device}')

## 1. LLM Ranking Paradigms

LLMs can rank items for recommendation in three main ways:

### Pointwise Ranking
Score each item independently: "Rate how relevant this item is to the user on a scale of 1-10."

### Pairwise Ranking
Compare items in pairs: "Which item is more relevant to the user: Item A or Item B?"

### Listwise Ranking
Rank an entire list at once: "Rank these 10 items from most to least relevant."

| Method | Complexity | Consistency | Scalability |
|--------|-----------|-------------|-------------|
| Pointwise | $O(n)$ calls | Low (uncalibrated) | Best |
| Pairwise | $O(n^2)$ calls | High | Worst |
| Listwise | $O(1)$ calls | Medium | Good (if list fits context) |

> **💡 Concept:** Pairwise ranking is most consistent (no calibration needed) but quadratic in cost. Listwise is most efficient but suffers from position bias. Pointwise scales best but needs calibration.

In [ ]:
# Create synthetic item data with relevance signals
n_items = 50
n_queries = 100  # Simulating user queries/sessions

categories = ['Electronics', 'Books', 'Clothing', 'Food', 'Sports']
item_data = []
for i in range(n_items):
    item_data.append({
        'id': i,
        'name': f'Product_{i}',
        'category': categories[i % len(categories)],
        'quality': np.random.uniform(0.3, 1.0),
        'popularity': np.random.uniform(0.1, 1.0),
        'price': np.random.uniform(10, 200)
    })

# Create query-item relevance scores (ground truth)
query_preferences = np.random.randn(n_queries, len(categories)) * 0.5
item_category_idx = np.array([categories.index(item['category']) for item in item_data])

# True relevance = category preference + quality + noise
true_relevance = np.zeros((n_queries, n_items))
for q in range(n_queries):
    for i in range(n_items):
        cat_match = query_preferences[q, item_category_idx[i]]
        true_relevance[q, i] = (
            cat_match * 2.0 +
            item_data[i]['quality'] * 1.5 +
            item_data[i]['popularity'] * 0.5 +
            np.random.randn() * 0.3
        )

print(f'Created {n_items} items across {len(categories)} categories')
print(f'Created {n_queries} queries with ground-truth relevance')
print(f'Sample item: {item_data[0]}')

## 2. Simulated LLM Ranker

We simulate an LLM ranker using a neural network that approximates LLM behavior — it takes text-like features and produces relevance judgments with realistic biases.

In [ ]:
class SimulatedLLMRanker:
    """Simulates LLM ranking behavior including typical biases."""
    
    def __init__(self, position_bias_strength=0.3, noise_level=0.5, seed=42):
        self.rng = np.random.RandomState(seed)
        self.position_bias_strength = position_bias_strength
        self.noise_level = noise_level
    
    def pointwise_score(self, query_idx, item_indices, true_relevance):
        """Score each item independently."""
        scores = []
        for item_idx in item_indices:
            # LLM approximates true relevance with noise
            score = true_relevance[query_idx, item_idx] + self.rng.randn() * self.noise_level
            # Quantize to 1-10 scale (typical LLM output)
            score = np.clip(score * 2 + 5, 1, 10)
            scores.append(score)
        return np.array(scores)
    
    def pairwise_compare(self, query_idx, item_a, item_b, true_relevance):
        """Compare two items. Returns True if A is preferred over B."""
        rel_a = true_relevance[query_idx, item_a] + self.rng.randn() * self.noise_level * 0.5
        rel_b = true_relevance[query_idx, item_b] + self.rng.randn() * self.noise_level * 0.5
        return rel_a > rel_b
    
    def listwise_rank(self, query_idx, item_indices, true_relevance):
        """Rank a list of items. Includes position bias."""
        n = len(item_indices)
        scores = np.zeros(n)
        
        for pos, item_idx in enumerate(item_indices):
            # True signal
            signal = true_relevance[query_idx, item_idx]
            # Position bias: items at the beginning and end get attention boost
            primacy_bias = self.position_bias_strength * np.exp(-pos / 3)
            recency_bias = self.position_bias_strength * np.exp(-(n - 1 - pos) / 5)
            position_bias = primacy_bias + recency_bias
            # Noise
            noise = self.rng.randn() * self.noise_level
            scores[pos] = signal + position_bias + noise
        
        # Return ranking (indices sorted by score, descending)
        ranking = np.argsort(-scores)
        return ranking


llm_ranker = SimulatedLLMRanker(position_bias_strength=0.3, noise_level=0.5)
print('LLM Ranker initialized.')

## 3. Comparing Ranking Strategies

In [ ]:
def ndcg_at_k(true_rel, ranking, k=10):
    """Compute NDCG@K given relevance scores and a ranking."""
    dcg = sum(true_rel[ranking[i]] / np.log2(i + 2) for i in range(min(k, len(ranking))))
    ideal_order = np.argsort(-true_rel)
    idcg = sum(true_rel[ideal_order[i]] / np.log2(i + 2) for i in range(min(k, len(true_rel))))
    return dcg / idcg if idcg > 0 else 0


def evaluate_ranking_methods(llm_ranker, true_relevance, n_candidates=20, k=10):
    """Compare pointwise, pairwise, and listwise ranking."""
    n_queries = true_relevance.shape[0]
    n_items = true_relevance.shape[1]
    
    results = {'pointwise': [], 'pairwise': [], 'listwise': [], 'random': [], 'oracle': []}
    
    for q in range(min(n_queries, 50)):  # Evaluate on 50 queries
        # Sample candidate items
        candidates = np.random.choice(n_items, n_candidates, replace=False)
        true_rel = true_relevance[q, candidates]
        # Normalize to [0, 1] for NDCG
        true_rel_norm = (true_rel - true_rel.min()) / (true_rel.max() - true_rel.min() + 1e-8)
        
        # --- Pointwise ---
        pw_scores = llm_ranker.pointwise_score(q, candidates, true_relevance)
        pw_ranking = np.argsort(-pw_scores)
        results['pointwise'].append(ndcg_at_k(true_rel_norm, pw_ranking, k))
        
        # --- Pairwise (bubble sort style) ---
        pair_order = list(range(n_candidates))
        # Simple pairwise: compare adjacent and swap
        for _ in range(3):  # Multiple passes
            for j in range(len(pair_order) - 1):
                if llm_ranker.pairwise_compare(q, candidates[pair_order[j+1]],
                                                candidates[pair_order[j]], true_relevance):
                    pair_order[j], pair_order[j+1] = pair_order[j+1], pair_order[j]
        results['pairwise'].append(ndcg_at_k(true_rel_norm, pair_order, k))
        
        # --- Listwise ---
        lw_ranking = llm_ranker.listwise_rank(q, candidates, true_relevance)
        results['listwise'].append(ndcg_at_k(true_rel_norm, lw_ranking, k))
        
        # --- Random ---
        random_ranking = np.random.permutation(n_candidates)
        results['random'].append(ndcg_at_k(true_rel_norm, random_ranking, k))
        
        # --- Oracle ---
        oracle_ranking = np.argsort(-true_rel_norm)
        results['oracle'].append(ndcg_at_k(true_rel_norm, oracle_ranking, k))
    
    return {k: np.mean(v) for k, v in results.items()}


results = evaluate_ranking_methods(llm_ranker, true_relevance)

print('NDCG@10 Results:')
for method, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f'  {method:12s}: {score:.4f}')

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
methods = list(results.keys())
scores = [results[m] for m in methods]
colors = ['gold', 'steelblue', 'coral', 'forestgreen', 'lightgray']
bars = ax.bar(methods, scores, color=colors)
ax.set_ylabel('NDCG@10')
ax.set_title('LLM Ranking Strategies Comparison')
ax.grid(True, alpha=0.3, axis='y')
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{score:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 4. Position Bias Analysis

LLMs exhibit **position bias** when ranking items in a list:
- **Primacy effect**: Items listed first get disproportionate attention
- **Recency effect**: Items listed last also get some attention boost
- **Lost in the middle**: Items in the middle of the list are often overlooked

> **⚠️ Common Pitfall:** Position bias means the order you present candidates to the LLM affects the ranking output! Always shuffle candidates and ideally average over multiple orderings.

In [ ]:
def analyze_position_bias(llm_ranker, true_relevance, n_trials=200, list_size=15):
    """Measure how input position affects output ranking."""
    n_items = true_relevance.shape[1]
    
    # Track: for each input position, what output rank does it get?
    position_to_rank = defaultdict(list)
    
    for trial in range(n_trials):
        q = trial % true_relevance.shape[0]
        # Use items with SIMILAR true relevance to isolate position effect
        candidates = np.random.choice(n_items, list_size, replace=False)
        
        ranking = llm_ranker.listwise_rank(q, candidates, true_relevance)
        
        for input_pos in range(list_size):
            output_rank = np.where(ranking == input_pos)[0][0]
            position_to_rank[input_pos].append(output_rank)
    
    # Compute average output rank per input position
    avg_ranks = {pos: np.mean(ranks) for pos, ranks in position_to_rank.items()}
    return avg_ranks


bias_results = analyze_position_bias(llm_ranker, true_relevance)

positions = sorted(bias_results.keys())
avg_output_ranks = [bias_results[p] for p in positions]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Position bias curve
axes[0].plot(positions, avg_output_ranks, 'b-o', markersize=6)
axes[0].axhline(y=np.mean(avg_output_ranks), color='r', linestyle='--', alpha=0.5, label='Expected (no bias)')
axes[0].set_xlabel('Input Position', fontsize=12)
axes[0].set_ylabel('Average Output Rank (lower=better)', fontsize=12)
axes[0].set_title('Position Bias in LLM Listwise Ranking', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].invert_yaxis()

# Plot 2: Debiasing via shuffling
# Compare single-pass vs multi-pass with shuffling
n_shuffles = [1, 3, 5, 10]
shuffle_ndcgs = []

for n_shuf in n_shuffles:
    ndcgs = []
    for q in range(50):
        candidates = np.random.choice(n_items, 20, replace=False)
        true_rel = true_relevance[q, candidates]
        true_rel_norm = (true_rel - true_rel.min()) / (true_rel.max() - true_rel.min() + 1e-8)
        
        # Average rankings over multiple shuffles
        rank_scores = np.zeros(len(candidates))
        for _ in range(n_shuf):
            perm = np.random.permutation(len(candidates))
            shuffled_candidates = candidates[perm]
            ranking = llm_ranker.listwise_rank(q, shuffled_candidates, true_relevance)
            for rank, pos in enumerate(ranking):
                original_pos = perm[pos]
                rank_scores[original_pos] += (len(candidates) - rank)
        
        final_ranking = np.argsort(-rank_scores)
        ndcgs.append(ndcg_at_k(true_rel_norm, final_ranking, 10))
    
    shuffle_ndcgs.append(np.mean(ndcgs))

axes[1].plot(n_shuffles, shuffle_ndcgs, 'g-o', markersize=8)
axes[1].set_xlabel('Number of Shuffles', fontsize=12)
axes[1].set_ylabel('NDCG@10', fontsize=12)
axes[1].set_title('Debiasing via Input Shuffling', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Calibration Analysis

LLM pointwise scores are often poorly calibrated — the absolute scores don't reflect true relevance probabilities.

In [ ]:
def calibration_analysis(llm_ranker, true_relevance, n_queries=50, n_candidates=30):
    """Analyze calibration of pointwise LLM scores."""
    all_scores = []
    all_true_rel = []
    
    for q in range(n_queries):
        candidates = np.random.choice(true_relevance.shape[1], n_candidates, replace=False)
        pw_scores = llm_ranker.pointwise_score(q, candidates, true_relevance)
        true_rel = true_relevance[q, candidates]
        all_scores.extend(pw_scores)
        all_true_rel.extend(true_rel)
    
    return np.array(all_scores), np.array(all_true_rel)


pw_scores, true_rel = calibration_analysis(llm_ranker, true_relevance)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter plot
axes[0].scatter(true_rel, pw_scores, alpha=0.3, s=10, color='steelblue')
axes[0].set_xlabel('True Relevance', fontsize=12)
axes[0].set_ylabel('LLM Pointwise Score', fontsize=12)
axes[0].set_title('Calibration: LLM Scores vs True Relevance', fontsize=13)
# Add trend line
z = np.polyfit(true_rel, pw_scores, 1)
p = np.poly1d(z)
x_line = np.linspace(true_rel.min(), true_rel.max(), 100)
axes[0].plot(x_line, p(x_line), 'r-', linewidth=2, label=f'Trend (r={np.corrcoef(true_rel, pw_scores)[0,1]:.3f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Score distribution
axes[1].hist(pw_scores, bins=30, alpha=0.7, color='steelblue', label='LLM Scores', density=True)
axes[1].set_xlabel('Score', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Distribution of LLM Pointwise Scores', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Rank correlation
from scipy.stats import spearmanr, kendalltau
spearman, _ = spearmanr(true_rel, pw_scores)
kendall, _ = kendalltau(true_rel, pw_scores)
print(f'Spearman correlation: {spearman:.4f}')
print(f'Kendall tau: {kendall:.4f}')

## 6. Chat-Rec: Conversational Ranking

**Chat-Rec** (Gao et al., 2023) frames recommendation as a conversation where the LLM interactively refines rankings based on user feedback.

> **🔑 Pro Tip:** Chat-Rec works best when combined with a retrieval stage. First retrieve 50-100 candidates with a traditional model, then use the LLM to re-rank the top candidates in conversation.

In [ ]:
class ChatRecSimulator:
    """Simulates Chat-Rec style conversational ranking."""
    
    def __init__(self, item_data, true_relevance):
        self.items = item_data
        self.true_rel = true_relevance
    
    def initial_ranking(self, query_idx, candidates, ranker):
        """Initial ranking without user feedback."""
        scores = ranker.pointwise_score(query_idx, candidates, self.true_rel)
        return candidates[np.argsort(-scores)]
    
    def user_feedback(self, query_idx, shown_item, true_relevance):
        """Simulate user feedback on a shown item."""
        rel = true_relevance[query_idx, shown_item]
        if rel > 1.0:
            return 'like', self.items[shown_item]['category']
        elif rel > -0.5:
            return 'neutral', self.items[shown_item]['category']
        else:
            return 'dislike', self.items[shown_item]['category']
    
    def re_rank_with_feedback(self, query_idx, candidates, feedback_history, ranker):
        """Re-rank based on accumulated feedback."""
        scores = ranker.pointwise_score(query_idx, candidates, self.true_rel)
        
        # Adjust scores based on feedback
        liked_categories = set()
        disliked_categories = set()
        for sentiment, category in feedback_history:
            if sentiment == 'like':
                liked_categories.add(category)
            elif sentiment == 'dislike':
                disliked_categories.add(category)
        
        adjusted_scores = scores.copy()
        for i, item_idx in enumerate(candidates):
            cat = self.items[item_idx]['category']
            if cat in liked_categories:
                adjusted_scores[i] += 1.5  # Boost liked categories
            if cat in disliked_categories:
                adjusted_scores[i] -= 2.0  # Penalize disliked categories
        
        return candidates[np.argsort(-adjusted_scores)]


# Simulate a conversational session
chat_rec = ChatRecSimulator(item_data, true_relevance)
query_idx = 0
candidates = np.random.choice(n_items, 20, replace=False)

print('=== Conversational Ranking Session ===')
print(f'Query {query_idx}: Initial candidates ({len(candidates)} items)\n')

# Turn 1: Initial ranking
ranking = chat_rec.initial_ranking(query_idx, candidates, llm_ranker)
true_rel = true_relevance[query_idx, ranking]
true_rel_norm = (true_rel - true_rel.min()) / (true_rel.max() - true_rel.min() + 1e-8)
ndcg_turn1 = ndcg_at_k(true_rel_norm, np.arange(len(ranking)), 10)
print(f'Turn 1 (no feedback) NDCG@10: {ndcg_turn1:.4f}')
print(f'Top 3: {[item_data[i]["name"] + " (" + item_data[i]["category"] + ")" for i in ranking[:3]]}\n')

# Simulate feedback turns
feedback_history = []
ndcg_by_turn = [ndcg_turn1]

for turn in range(1, 4):
    # Show top item and get feedback
    shown = ranking[0]
    sentiment, category = chat_rec.user_feedback(query_idx, shown, true_relevance)
    feedback_history.append((sentiment, category))
    print(f'Turn {turn+1}: User {sentiment}s {item_data[shown]["name"]} ({category})')
    
    # Remove shown item and re-rank
    remaining = np.array([c for c in candidates if c != shown])
    ranking = chat_rec.re_rank_with_feedback(query_idx, remaining, feedback_history, llm_ranker)
    
    true_rel = true_relevance[query_idx, ranking]
    true_rel_norm = (true_rel - true_rel.min()) / (true_rel.max() - true_rel.min() + 1e-8)
    ndcg = ndcg_at_k(true_rel_norm, np.arange(len(ranking)), 10)
    ndcg_by_turn.append(ndcg)
    print(f'  Updated NDCG@10: {ndcg:.4f}')
    print(f'  New top 3: {[item_data[i]["name"] + " (" + item_data[i]["category"] + ")" for i in ranking[:3]]}\n')

# Plot improvement over turns
plt.figure(figsize=(8, 4))
plt.plot(range(len(ndcg_by_turn)), ndcg_by_turn, 'g-o', markersize=8)
plt.xlabel('Conversation Turn', fontsize=12)
plt.ylabel('NDCG@10', fontsize=12)
plt.title('Chat-Rec: Ranking Quality Improves with Feedback', fontsize=13)
plt.grid(True, alpha=0.3)
plt.xticks(range(len(ndcg_by_turn)))
plt.tight_layout()
plt.show()

## Exercises

### 🏋️ Exercise 1: Implement Pairwise Tournament Ranking

Implement a more efficient pairwise approach using tournament-style elimination.

In [ ]:
# TODO: Implement tournament-style pairwise ranking
def tournament_rank(llm_ranker, query_idx, candidates, true_relevance):
    """
    TODO:
    1. Organize candidates into pairs
    2. Run pairwise comparisons (winner advances)
    3. Use multiple rounds to build a complete ranking
    4. Compare: number of LLM calls vs bubble sort approach
    5. Evaluate NDCG@10
    """
    pass

# TODO: Compare tournament vs bubble sort pairwise in terms of:
# - Number of comparisons needed
# - NDCG@10 quality
# - Plot the trade-off

### 🏋️ Exercise 2: Implement Score Calibration

Build a calibration module that transforms LLM pointwise scores into well-calibrated relevance estimates.

In [ ]:
# TODO: Implement calibration for LLM pointwise scores
class ScoreCalibrator:
    """
    TODO:
    1. Collect (LLM_score, true_relevance) pairs on validation set
    2. Fit isotonic regression or Platt scaling
    3. Apply calibration to test scores
    4. Compare NDCG before and after calibration
    5. Show calibration plot (expected vs observed)
    """
    def fit(self, llm_scores, true_relevance):
        pass
    
    def calibrate(self, llm_scores):
        pass

### 🏋️ Exercise 3: Sliding Window Listwise Ranking

For large candidate sets, implement a sliding window approach.

In [ ]:
# TODO: Sliding window listwise ranking
def sliding_window_rank(llm_ranker, query_idx, candidates, true_relevance,
                         window_size=10, stride=5):
    """
    TODO: For candidate lists larger than the LLM context:
    1. Process candidates in overlapping windows of `window_size`
    2. Get local rankings within each window
    3. Merge local rankings using a scoring scheme
    4. Handle items that appear in multiple windows
    """
    pass

# TODO: Compare with single-pass listwise on 50-item candidate lists
# TODO: Plot NDCG vs window_size

## Summary

| Approach | Best For | Key Challenge | Reference |
|----------|----------|--------------|----------|
| **Pointwise** | Large candidate sets | Poor calibration | — |
| **Pairwise** | High-precision top-K | Quadratic cost | — |
| **Listwise** | Small candidate sets | Position bias | — |
| **Chat-Rec** | Interactive settings | Multi-turn complexity | Gao et al., 2023 |

**Key Takeaways:**
1. No single ranking strategy dominates — the choice depends on the scenario
2. Position bias is a real issue in listwise ranking — shuffle and average
3. LLM scores need calibration for pointwise ranking
4. Conversational ranking improves with user feedback but adds latency
5. Hybrid approaches (traditional retrieval + LLM re-ranking) are most practical